# TimesNet on Server - Setup and Training

This notebook contains all commands needed to run TimesNet on the remote server.

## Check GPU Availability

In [ ]:
!nvidia-smi

## Check Current Directory

In [ ]:
!pwd
!ls -la

## Navigate to Project Directory

In [15]:
%cd /home/fzf/dev/TSAD/TimesNet/Time-Series-Library

/home/fzf/dev/TSAD/TimesNet/Time-Series-Library


/home/fzf/.conda/envs/timesnet/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## Setup Conda Environment (First Time Only)

Run these cells only once to create the environment:

In [ ]:
# Create conda environment (only run once)
!conda create -n timesnet python=3.9 -y

In [ ]:
# Register environment as Jupyter kernel (only run once)
!conda run -n timesnet pip install ipykernel
!conda run -n timesnet python -m ipykernel install --user --name=timesnet --display-name "Python (timesnet)"

## Install Dependencies

**IMPORTANT:** After running the cells above, switch the kernel to "Python (timesnet)" using the kernel selector in the top right!

Then run these cells to install packages:

In [2]:
# Verify we're using the correct environment
import sys
print(f"Python path: {sys.executable}")
print(f"Should contain 'timesnet' in the path")

Python path: /home/fzf/.conda/envs/timesnet/bin/python
Should contain 'timesnet' in the path


In [ ]:
# Install PyTorch with CUDA support
!pip install torch torchvision torchaudio

In [ ]:
# Install other dependencies
!pip install einops reformer-pytorch local-attention sktime sympy PyWavelets patool tqdm huggingface_hub datasets pandas numpy matplotlib scikit-learn

In [ ]:
# Install additional requirements if requirements.txt exists
!pip install -r requirements.txt

## Verify GPU is Available in Python

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

## Run TimesNet

Choose which GPU to use by setting CUDA_VISIBLE_DEVICES:

In [1]:
# Set which GPU to use (0, 1, 2, etc.)
# IMPORTANT: This must run BEFORE any torch.cuda call (e.g. the "Verify GPU" cell).
# If CUDA is already initialized, restart the kernel first.
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'  # Only expose physical GPU 3 → cuda:0

## SMD Dataset

In [4]:
!python -u run.py \
  --task_name anomaly_detection \
  --is_training 0 \
  --root_path ./dataset/SMDT \
  --model_id SMD \
  --model TimesNet \
  --data SMD \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 64 \
  --d_ff 64 \
  --e_layers 3 \
  --enc_in 38 \
  --c_out 38 \
  --top_k 3 \
  --anomaly_ratio 0.5 \
  --batch_size 128 \
  --train_epochs 10

Using GPU
Args in experiment:
Basic Config
  Task Name:          anomaly_detection   Is Training:        0                   
  Model ID:           SMD                 Model:              TimesNet            

Data Loader
  Data:               SMD                 Root Path:          ./dataset/SMDT      
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Anomaly Detection Task
  Anomaly Ratio:      0.5                 

Model Parameters
  Top k:              3                   Num Kernels:        6                   
  Enc In:             38                  Dec In:             7                   
  C Out:              38                  d model:            64                  
  n heads:            8                   e layers:           3                   
  d layers:           1                   d FF:               64     

In [2]:
import numpy as np
import os

def get_anomaly_segments(labels):
    """Return list of (start, end) for each contiguous run of 1s (end is exclusive)."""
    segments = []
    in_seg = False
    start = 0
    for i, v in enumerate(labels):
        if v == 1 and not in_seg:
            in_seg = True
            start = i
        elif v == 0 and in_seg:
            in_seg = False
            segments.append((start, i))
    if in_seg:
        segments.append((start, len(labels)))
    return segments

def analyze_segments(folder_path):
    gt        = np.load(os.path.join(folder_path, 'ground_truth.npy'))
    pred_raw  = np.load(os.path.join(folder_path, 'predictions_raw.npy'))
    scores    = np.load(os.path.join(folder_path, 'anomaly_scores.npy'))
    threshold = np.load(os.path.join(folder_path, 'threshold.npy'))[0]

    segments = get_anomaly_segments(gt)
    results = []
    for start, end in segments:
        results.append({
            'start':           start,
            'end':             end,
            'length':          end - start,
            'detected':        bool(pred_raw[start:end].any()),
            'max_score':       float(scores[start:end].max()),
            'mean_score':      float(scores[start:end].mean()),
            'score_ratio':     float(scores[start:end].max() / threshold),
        })
    return results, threshold

base = './test_results'
folders = sorted([
    f for f in os.listdir(base)
    if os.path.isdir(os.path.join(base, f))
    and os.path.exists(os.path.join(base, f, 'ground_truth.npy'))
])

for folder in folders:
    folder_path = os.path.join(base, folder)
    dataset = folder.split('_')[2]  # e.g. "SMD", "MSL", ...

    results, threshold = analyze_segments(folder_path)
    if not results:
        print(f"{dataset}: no anomaly segments in ground truth")
        continue

    total    = len(results)
    detected = sum(r['detected'] for r in results)
    missed   = total - detected

    print(f"\n{'='*65}")
    print(f"Dataset: {dataset}   threshold: {threshold:.4f}")
    print(f"Anomaly segments — total: {total}  detected: {detected}  missed: {missed}  "
          f"segment-recall: {detected/total:.1%}")

    if missed:
        missed_segs = sorted([r for r in results if not r['detected']],
                             key=lambda x: x['max_score'], reverse=True)
        print(f"\n  MISSED ({missed}) — sorted by max reconstruction error:")
        print(f"  {'start':>8}  {'end':>8}  {'len':>6}  {'max_score':>10}  {'score/thresh':>13}")
        for r in missed_segs[:25]:
            print(f"  {r['start']:>8}  {r['end']:>8}  {r['length']:>6}  "
                  f"{r['max_score']:>10.4f}  {r['score_ratio']:>12.3f}x")
        if len(missed_segs) > 25:
            print(f"  ... and {len(missed_segs)-25} more")

    det_segs = sorted([r for r in results if r['detected']],
                      key=lambda x: x['max_score'], reverse=True)
    print(f"\n  DETECTED ({detected}) — top 10 by max reconstruction error:")
    print(f"  {'start':>8}  {'end':>8}  {'len':>6}  {'max_score':>10}  {'score/thresh':>13}")
    for r in det_segs[:10]:
        print(f"  {r['start']:>8}  {r['end']:>8}  {r['length']:>6}  "
              f"{r['max_score']:>10.4f}  {r['score_ratio']:>12.3f}x")


Dataset: SMD   threshold: 2.8342
Anomaly segments — total: 327  detected: 217  missed: 110  segment-recall: 66.4%

  MISSED (110) — sorted by max reconstruction error:
     start       end     len   max_score   score/thresh
    549765    549837      72      2.8094         0.991x
    264652    264740      88      2.7329         0.964x
    550594    550616      22      2.7294         0.963x
    274325    274328       3      2.7154         0.958x
    140638    140664      26      2.6471         0.934x
    467581    467685     104      2.6060         0.919x
    280065    280076      11      2.5782         0.910x
     35805     35833      28      2.5559         0.902x
     29736     29740       4      2.4920         0.879x
    193592    193648      56      2.3971         0.846x
    167873    167989     116      2.3729         0.837x
    134567    134570       3      2.3723         0.837x
     14068     14072       4      2.3183         0.818x
    159707    159710       3      2.3004       

In [ ]:
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')   # no display needed on server; remove this line if running locally
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── helpers (duplicated here so this cell is self-contained) ──────────

def get_anomaly_segments(labels):
    segments, in_s, s0 = [], False, 0
    for i, v in enumerate(labels):
        if v == 1 and not in_s:
            in_s, s0 = True, i
        elif v == 0 and in_s:
            in_s = False
            segments.append((s0, i))
    if in_s:
        segments.append((s0, len(labels)))
    return segments

def load_result(folder_path):
    gt        = np.load(os.path.join(folder_path, 'ground_truth.npy'))
    pred_raw  = np.load(os.path.join(folder_path, 'predictions_raw.npy'))
    scores    = np.load(os.path.join(folder_path, 'anomaly_scores.npy'))
    threshold = float(np.load(os.path.join(folder_path, 'threshold.npy'))[0])
    return gt, pred_raw, scores, threshold

# ── config ────────────────────────────────────────────────────────────

CONTEXT        = 500    # time steps of context on each side of a zoom window
MAX_ZOOM       = 8      # max missed segments to zoom into per dataset
MAX_OV_POINTS  = 40000  # downsample overview to this many points for speed
OUT_DIR        = './analysis_plots'
os.makedirs(OUT_DIR, exist_ok=True)

# ── shared colours / style ────────────────────────────────────────────

C_SCORE     = '#2c7bb6'
C_THRESH    = '#333333'
C_DETECTED  = '#2ca02c'
C_MISSED    = '#d62728'
C_PRED      = '#ff7f0e'

base = './test_results'
folders = sorted([
    f for f in os.listdir(base)
    if os.path.isdir(os.path.join(base, f))
    and os.path.exists(os.path.join(base, f, 'ground_truth.npy'))
])

for folder in folders:
    folder_path = os.path.join(base, folder)
    dataset     = folder.split('_')[2]

    gt, pred_raw, scores, threshold = load_result(folder_path)
    segments = get_anomaly_segments(gt)
    if not segments:
        print(f"{dataset}: no anomaly segments — skipping")
        continue

    detected_segs = [(s, e) for s, e in segments if pred_raw[s:e].any()]
    missed_segs   = [(s, e) for s, e in segments if not pred_raw[s:e].any()]

    print(f"\n{dataset}: {len(segments)} segments  "
          f"detected={len(detected_segs)}  missed={len(missed_segs)}")

    # ── FIGURE 1: overview ────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(20, 4))

    # downsample score line for large datasets
    step = max(1, len(scores) // MAX_OV_POINTS)
    t    = np.arange(0, len(scores), step)
    ax.plot(t, scores[::step], lw=0.5, color=C_SCORE, zorder=2)
    ax.axhline(threshold, color=C_THRESH, lw=1.2, ls='--', zorder=3,
               label=f'Threshold ({threshold:.3f})')

    for s, e in detected_segs:
        ax.axvspan(s, e, color=C_DETECTED, alpha=0.30, lw=0)
    for s, e in missed_segs:
        ax.axvspan(s, e, color=C_MISSED,   alpha=0.45, lw=0)

    ax.set_yscale('log')
    ax.set_xlim(0, len(scores))
    ax.set_xlabel('Time step')
    ax.set_ylabel('Reconstruction error (log)')
    ax.set_title(f'{dataset} — reconstruction error over time   '
                 f'({len(detected_segs)} detected, {len(missed_segs)} missed)')

    legend_handles = [
        mpatches.Patch(color=C_DETECTED, alpha=0.5, label=f'Detected ({len(detected_segs)})'),
        mpatches.Patch(color=C_MISSED,   alpha=0.6, label=f'Missed ({len(missed_segs)})'),
        plt.Line2D([0], [0], color=C_THRESH, ls='--', lw=1.2, label=f'Threshold'),
        plt.Line2D([0], [0], color=C_SCORE,  lw=1,   label='Recon. error'),
    ]
    ax.legend(handles=legend_handles, loc='upper left', fontsize=8)
    plt.tight_layout()

    out = os.path.join(OUT_DIR, f'{dataset}_overview.png')
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"  saved {out}")

    # ── FIGURE 2: zoom into missed segments ───────────────────────────
    if not missed_segs:
        continue

    # sort by segment length descending — longest missed = most anomaly time lost
    zoom_segs = sorted(missed_segs, key=lambda x: x[1] - x[0], reverse=True)[:MAX_ZOOM]
    n = len(zoom_segs)

    fig, axes = plt.subplots(n, 1, figsize=(18, 3.5 * n), squeeze=False)

    for i, (s, e) in enumerate(zoom_segs):
        ax  = axes[i, 0]
        lo  = max(0, s - CONTEXT)
        hi  = min(len(scores), e + CONTEXT)
        t   = np.arange(lo, hi)

        ax.plot(t, scores[lo:hi], lw=0.8, color=C_SCORE, zorder=2, label='Recon. error')
        ax.axhline(threshold, color=C_THRESH, lw=1.2, ls='--', zorder=3, label='Threshold')

        # shade every gt segment visible in this window
        for seg_s, seg_e in segments:
            if seg_e <= lo or seg_s >= hi:
                continue
            color = C_DETECTED if pred_raw[seg_s:seg_e].any() else C_MISSED
            alpha = 0.25        if color == C_DETECTED          else 0.40
            ax.axvspan(max(seg_s, lo), min(seg_e, hi),
                       color=color, alpha=alpha, lw=0)

        # draw the raw prediction as a filled step at the bottom
        pred_win = pred_raw[lo:hi].astype(float)
        ymin, ymax = ax.get_ylim()
        # place the prediction band at the very bottom (will be rescaled after)
        ax.fill_between(t, 0, pred_win * threshold * 0.5,
                        step='mid', color=C_PRED, alpha=0.5, zorder=1,
                        label='Pred (raw, scaled)')

        ax.set_yscale('log')
        ax.set_xlim(lo, hi)
        ax.set_ylabel('Recon. error (log)')

        ratio = scores[s:e].max() / threshold
        ax.set_title(
            f'Missed  [{s} : {e}]   len={e - s}   '
            f'max_score={scores[s:e].max():.4f}   score/threshold={ratio:.3f}×',
            fontsize=9
        )
        if i == 0:
            ax.legend(loc='upper right', fontsize=7)

    axes[-1, 0].set_xlabel('Time step')
    fig.suptitle(f'{dataset} — missed segments (longest first)', fontsize=11, y=1.005)
    plt.tight_layout()

    out = os.path.join(OUT_DIR, f'{dataset}_missed_zoom.png')
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f"  saved {out}")


SMD: 327 segments  detected=217  missed=110
  saved ./analysis_plots/SMD_overview.png
  saved ./analysis_plots/SMD_missed_zoom.png


## Worst Missed Segment — Channel Detail

For each dataset, finds the longest missed anomaly segment, picks the **8 channels with the highest reconstruction error** in that segment, and plots them.

Each channel lane: **black = input**, **orange = reconstruction**, blue shading = true anomaly, red shading = predicted anomaly.  
Bottom lane: anomaly score + threshold.

Requires a re-run of the test script after the latest `exp_anomaly_detection.py` changes so that `test_input.npy`, `test_output.npy`, and `step_size.npy` exist in `test_results/<setting>/`.

In [6]:
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

WIN_SIZE = 100   # seq_len used during training
CONTEXT  = 500   # extra time steps shown on each side of the segment
MAX_CH   = 8
OUT_DIR  = './analysis_plots'
os.makedirs(OUT_DIR, exist_ok=True)

C_INPUT  = '#111111'
C_RECON  = '#ff7f0e'
C_GT     = '#1f77b4'   # blue  — true anomaly shading
C_PRED   = '#d62728'   # red   — predicted anomaly shading
C_SCORE  = '#2c7bb6'
C_THRESH = '#333333'

# ── helpers ───────────────────────────────────────────────────────────

def get_segments(labels):
    segs, in_s, s0 = [], False, 0
    for i, v in enumerate(labels):
        if v == 1 and not in_s:  in_s, s0 = True, i
        elif v == 0 and in_s:    in_s = False; segs.append((s0, i))
    if in_s: segs.append((s0, len(labels)))
    return segs

def shade_spans(ax, binary, t_offset, color, alpha):
    in_s = False
    for i, v in enumerate(binary):
        if v and not in_s:   in_s = True;  s0 = i
        elif not v and in_s: in_s = False; ax.axvspan(t_offset+s0, t_offset+i, color=color, alpha=alpha, lw=0)
    if in_s: ax.axvspan(t_offset+s0, t_offset+len(binary), color=color, alpha=alpha, lw=0)

# ── iterate over all result folders ───────────────────────────────────

base = './test_results'
folders = sorted([
    f for f in os.listdir(base)
    if os.path.isdir(os.path.join(base, f))
    and os.path.exists(os.path.join(base, f, 'test_input.npy'))
    and os.path.exists(os.path.join(base, f, 'test_output.npy'))
])

if not folders:
    print("No folders with test_input.npy / test_output.npy found.")
    print("Re-run the test script after the latest exp_anomaly_detection.py changes.")
else:
    for folder in folders:
        fp      = os.path.join(base, folder)
        dataset = folder.split('_')[2]

        gt        = np.load(os.path.join(fp, 'ground_truth.npy')).astype(int)
        pred_raw  = np.load(os.path.join(fp, 'predictions_raw.npy')).astype(int)
        scores    = np.load(os.path.join(fp, 'anomaly_scores.npy')).astype(np.float32)
        threshold = float(np.load(os.path.join(fp, 'threshold.npy'))[0])
        x_in      = np.load(os.path.join(fp, 'test_input.npy')).astype(np.float32)   # (T, C)
        x_out     = np.load(os.path.join(fp, 'test_output.npy')).astype(np.float32)  # (T, C)
        step_size = int(np.load(os.path.join(fp, 'step_size.npy'))[0]) \
                    if os.path.exists(os.path.join(fp, 'step_size.npy')) else WIN_SIZE

        # ── convert to raw time space ─────────────────────────────────
        # step=100 (SMD): window-space == raw-space, no subsampling needed
        # step=1  (rest): take every WIN_SIZE-th point (position-0 of each window)
        sub = 1 if step_size == WIN_SIZE else WIN_SIZE
        gt_r     = gt[::sub]
        pred_r   = pred_raw[::sub]
        scores_r = scores[::sub]
        x_in_r   = x_in[::sub]    # (T_raw, C)
        x_out_r  = x_out[::sub]   # (T_raw, C)

        # ── find the longest missed segment ───────────────────────────
        segments   = get_segments(gt_r)
        missed     = [(s, e) for s, e in segments if not pred_r[s:e].any()]
        if not missed:
            print(f"{dataset}: no missed segments — skipping")
            continue
        s, e = max(missed, key=lambda x: x[1] - x[0])
        print(f"\n{dataset}: worst missed segment [{s}:{e}] len={e-s}  "
              f"max_score={scores_r[s:e].max():.4f}  threshold={threshold:.4f}")

        # ── pick top MAX_CH channels by mean squared error in segment ─
        seg_err = np.mean((x_in_r[s:e] - x_out_r[s:e]) ** 2, axis=0)   # (C,)
        top_ch  = np.argsort(seg_err)[-MAX_CH:][::-1]
        print(f"  top channels (highest recon error): {top_ch.tolist()}")

        # ── plot window ───────────────────────────────────────────────
        lo = max(0, s - CONTEXT)
        hi = min(len(scores_r), e + CONTEXT)
        t  = np.arange(lo, hi)

        n_rows = len(top_ch) + 1   # channels + score lane
        fig, axes = plt.subplots(n_rows, 1,
                                 figsize=(20, 2.4 * n_rows),
                                 sharex=True, squeeze=False,
                                 gridspec_kw={'height_ratios': [1]*len(top_ch) + [1.4]})
        axes = axes[:, 0]

        gt_win   = gt_r[lo:hi]
        pred_win = pred_r[lo:hi]

        for row, ch in enumerate(top_ch):
            ax = axes[row]

            shade_spans(ax, gt_win,   lo, C_GT,   alpha=0.20)
            shade_spans(ax, pred_win, lo, C_PRED,  alpha=0.15)

            ax.plot(t, x_in_r[lo:hi,  ch], color=C_INPUT, lw=0.9,
                    label='Input'          if row == 0 else '_')
            ax.plot(t, x_out_r[lo:hi, ch], color=C_RECON, lw=0.9, ls='--',
                    label='Reconstruction' if row == 0 else '_')

            ax.set_ylabel(f'ch {ch}', fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(True, alpha=0.12)

        # ── score lane ────────────────────────────────────────────────
        ax_s = axes[-1]
        shade_spans(ax_s, gt_win,   lo, C_GT,   alpha=0.20)
        shade_spans(ax_s, pred_win, lo, C_PRED,  alpha=0.15)

        ax_s.plot(t, scores_r[lo:hi], color=C_SCORE, lw=0.9, label='Anomaly score')
        ax_s.axhline(threshold, color=C_THRESH, lw=1.2, ls='--',
                     label=f'Threshold ({threshold:.4g})')
        ax_s.set_yscale('log')
        ax_s.set_ylabel('Score (log)', fontsize=8)
        ax_s.set_xlabel('Time step')
        ax_s.tick_params(labelsize=7)
        ax_s.grid(True, alpha=0.12)

        # ── legend ────────────────────────────────────────────────────
        legend_handles = [
            plt.Line2D([0],[0], color=C_INPUT, lw=1.2,        label='Input'),
            plt.Line2D([0],[0], color=C_RECON, lw=1.2, ls='--', label='Reconstruction'),
            mpatches.Patch(color=C_GT,   alpha=0.35, label='True anomaly'),
            mpatches.Patch(color=C_PRED, alpha=0.35, label='Predicted anomaly'),
        ]
        axes[0].legend(handles=legend_handles, loc='upper right', fontsize=8, ncol=4)

        ratio = scores_r[s:e].max() / threshold
        fig.suptitle(
            f'{dataset} — worst missed segment  [{s}:{e}]  len={e-s}  '
            f'score/threshold={ratio:.3f}×\n'
            f'Top {len(top_ch)} channels by reconstruction error  '
            f'(blue = true anomaly,  red = predicted anomaly)',
            fontsize=10, y=1.002
        )
        plt.tight_layout()

        out = os.path.join(OUT_DIR, f'{dataset}_worst_segment_channels.png')
        fig.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f"  saved {out}")


SMD: worst missed segment [438865:439438] len=573  max_score=0.9082  threshold=2.8342
  top channels (highest recon error): [13, 12, 24, 1, 2, 15, 10, 18]
  saved ./analysis_plots/SMD_worst_segment_channels.png


In [ ]:
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── config — change these to taste ───────────────────────────────────
DATASET  = None   # e.g. 'SMD', 'MSL', 'SMAP', 'SWaT', 'PSM' — or None for all
CONTEXT  = 500    # extra time steps shown on each side of the segment
MAX_CH   = 8
WIN_SIZE = 100    # seq_len used during training
OUT_DIR  = './analysis_plots'
# ─────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)

C_INPUT  = '#111111'
C_RECON  = '#ff7f0e'
C_GT     = '#1f77b4'
C_PRED   = '#d62728'
C_SCORE  = '#2c7bb6'
C_THRESH = '#333333'

def get_segments(labels):
    segs, in_s, s0 = [], False, 0
    for i, v in enumerate(labels):
        if v == 1 and not in_s:  in_s, s0 = True, i
        elif v == 0 and in_s:    in_s = False; segs.append((s0, i))
    if in_s: segs.append((s0, len(labels)))
    return segs

def shade_spans(ax, binary, t_offset, color, alpha):
    in_s = False
    for i, v in enumerate(binary):
        if v and not in_s:   in_s = True;  s0 = i
        elif not v and in_s: in_s = False; ax.axvspan(t_offset+s0, t_offset+i, color=color, alpha=alpha, lw=0)
    if in_s: ax.axvspan(t_offset+s0, t_offset+len(binary), color=color, alpha=alpha, lw=0)

base = './test_results'
all_folders = sorted([
    f for f in os.listdir(base)
    if os.path.isdir(os.path.join(base, f))
    and os.path.exists(os.path.join(base, f, 'test_input.npy'))
    and os.path.exists(os.path.join(base, f, 'test_output.npy'))
])

# filter to the requested dataset if specified
folders = [f for f in all_folders if DATASET is None or f.split('_')[2] == DATASET]

if not folders:
    available = sorted({f.split('_')[2] for f in all_folders})
    print(f"No matching folders found for DATASET={DATASET!r}.")
    print(f"Available datasets with test_input/output: {available or 'none — re-run the test script first'}")
else:
    for folder in folders:
        fp      = os.path.join(base, folder)
        dataset = folder.split('_')[2]

        gt        = np.load(os.path.join(fp, 'ground_truth.npy')).astype(int)
        pred_raw  = np.load(os.path.join(fp, 'predictions_raw.npy')).astype(int)
        scores    = np.load(os.path.join(fp, 'anomaly_scores.npy')).astype(np.float32)
        threshold = float(np.load(os.path.join(fp, 'threshold.npy'))[0])
        x_in      = np.load(os.path.join(fp, 'test_input.npy')).astype(np.float32)
        x_out     = np.load(os.path.join(fp, 'test_output.npy')).astype(np.float32)

        # x_in/x_out are stored in raw time space (already subsampled by seq_len for step=1 datasets)
        # scores/gt/pred are in window-space; subsample them to match x_in/x_out length
        if len(x_in) < len(scores):
            sub      = len(scores) // len(x_in)
            gt_r     = gt[::sub]
            pred_r   = pred_raw[::sub]
            scores_r = scores[::sub]
        else:
            gt_r     = gt
            pred_r   = pred_raw
            scores_r = scores
        x_in_r  = x_in
        x_out_r = x_out

        segments = get_segments(gt_r)
        missed   = [(s, e) for s, e in segments if not pred_r[s:e].any()]
        if not missed:
            print(f"{dataset}: no missed segments — skipping")
            continue
        s, e = max(missed, key=lambda x: x[1] - x[0])
        print(f"\n{dataset}: worst missed segment [{s}:{e}] len={e-s}  "
              f"max_score={scores_r[s:e].max():.4f}  threshold={threshold:.4f}")

        seg_err = np.mean((x_in_r[s:e] - x_out_r[s:e]) ** 2, axis=0)
        top_ch  = np.argsort(seg_err)[-MAX_CH:][::-1]
        print(f"  top channels (highest recon error): {top_ch.tolist()}")

        lo = max(0, s - CONTEXT)
        hi = min(len(scores_r), e + CONTEXT)
        t  = np.arange(lo, hi)

        n_rows = len(top_ch) + 1
        fig, axes = plt.subplots(n_rows, 1,
                                 figsize=(20, 2.4 * n_rows),
                                 sharex=True, squeeze=False,
                                 gridspec_kw={'height_ratios': [1]*len(top_ch) + [1.4]})
        axes    = axes[:, 0]
        gt_win  = gt_r[lo:hi]
        pred_win = pred_r[lo:hi]

        for row, ch in enumerate(top_ch):
            ax = axes[row]
            shade_spans(ax, gt_win,   lo, C_GT,   alpha=0.20)
            shade_spans(ax, pred_win, lo, C_PRED,  alpha=0.15)
            ax.plot(t, x_in_r[lo:hi,  ch], color=C_INPUT, lw=0.9,
                    label='Input'          if row == 0 else '_')
            ax.plot(t, x_out_r[lo:hi, ch], color=C_RECON, lw=0.9,
                    label='Reconstruction' if row == 0 else '_')
            ax.set_ylabel(f'ch {ch}', fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(True, alpha=0.12)

        ax_s = axes[-1]
        shade_spans(ax_s, gt_win,   lo, C_GT,   alpha=0.20)
        shade_spans(ax_s, pred_win, lo, C_PRED,  alpha=0.15)
        ax_s.plot(t, scores_r[lo:hi], color=C_SCORE, lw=0.9, label='Anomaly score')
        ax_s.axhline(threshold, color=C_THRESH, lw=1.2, ls='--',
                     label=f'Threshold ({threshold:.4g})')
        ax_s.set_yscale('log')
        ax_s.set_ylabel('Score (log)', fontsize=8)
        ax_s.set_xlabel('Time step')
        ax_s.tick_params(labelsize=7)
        ax_s.grid(True, alpha=0.12)

        legend_handles = [
            plt.Line2D([0],[0], color=C_INPUT, lw=1.2, label='Input'),
            plt.Line2D([0],[0], color=C_RECON, lw=1.2, label='Reconstruction'),
            mpatches.Patch(color=C_GT,   alpha=0.35, label='True anomaly'),
            mpatches.Patch(color=C_PRED, alpha=0.35, label='Predicted anomaly'),
        ]
        axes[0].legend(handles=legend_handles, loc='upper right', fontsize=8, ncol=4)

        ratio = scores_r[s:e].max() / threshold
        fig.suptitle(
            f'{dataset} — worst missed segment  [{s}:{e}]  len={e-s}  '
            f'score/threshold={ratio:.3f}×\n'
            f'Top {len(top_ch)} channels by reconstruction error  '
            f'(blue = true anomaly,  red = predicted anomaly)',
            fontsize=10, y=1.002
        )
        plt.tight_layout()

        out = os.path.join(OUT_DIR, f'{dataset}_worst_segment_channels.png')
        fig.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f"  saved {out}")

## Custom Time Range — Channel Detail

Same channel-level visualization as the cell above, but instead of auto-selecting the worst missed segment you specify the window directly.

Set `T_START` and `T_END` to the raw-time-space indices you want to inspect (as printed by the segment analysis cell). `CONTEXT` adds optional padding on each side.

In [2]:
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── config — set T_START / T_END to the window you want to inspect ───
DATASET  = 'SMAP'   # e.g. 'SMD', 'MSL', 'SMAP', 'SWaT', 'PSM'
T_START  = 63560    # start index in raw time space  ← set this
T_END    = 67425    # end   index in raw time space  ← set this
CONTEXT  = 0       # extra steps shown on each side (0 = exact range)
MAX_CH   = 8
OUT_DIR  = './analysis_plots'
# ─────────────────────────────────────────────────────────────────────

if T_START is None or T_END is None:
    raise ValueError("Set T_START and T_END to the time range you want to plot.")

os.makedirs(OUT_DIR, exist_ok=True)

C_INPUT  = '#111111'
C_RECON  = '#ff7f0e'
C_GT     = '#1f77b4'
C_PRED   = '#d62728'
C_SCORE  = '#2c7bb6'
C_THRESH = '#333333'

def shade_spans(ax, binary, t_offset, color, alpha):
    in_s = False
    for i, v in enumerate(binary):
        if v and not in_s:   in_s = True;  s0 = i
        elif not v and in_s: in_s = False; ax.axvspan(t_offset+s0, t_offset+i, color=color, alpha=alpha, lw=0)
    if in_s: ax.axvspan(t_offset+s0, t_offset+len(binary), color=color, alpha=alpha, lw=0)

base = './test_results'
all_folders = sorted([
    f for f in os.listdir(base)
    if os.path.isdir(os.path.join(base, f))
    and os.path.exists(os.path.join(base, f, 'test_input.npy'))
    and os.path.exists(os.path.join(base, f, 'test_output.npy'))
])
folders = [f for f in all_folders if f.split('_')[2] == DATASET]

if not folders:
    available = sorted({f.split('_')[2] for f in all_folders})
    print(f"No folders for DATASET={DATASET!r}. Available: {available}")
else:
    for folder in folders:
        fp      = os.path.join(base, folder)
        dataset = folder.split('_')[2]

        gt        = np.load(os.path.join(fp, 'ground_truth.npy')).astype(int)
        pred_raw  = np.load(os.path.join(fp, 'predictions_raw.npy')).astype(int)
        scores    = np.load(os.path.join(fp, 'anomaly_scores.npy')).astype(np.float32)
        threshold = float(np.load(os.path.join(fp, 'threshold.npy'))[0])
        x_in      = np.load(os.path.join(fp, 'test_input.npy')).astype(np.float32)
        x_out     = np.load(os.path.join(fp, 'test_output.npy')).astype(np.float32)

        # x_in/x_out are in raw time space; subsample scores/gt/pred to match if needed
        if len(x_in) < len(scores):
            sub      = len(scores) // len(x_in)
            gt_r     = gt[::sub]
            pred_r   = pred_raw[::sub]
            scores_r = scores[::sub]
        else:
            gt_r     = gt
            pred_r   = pred_raw
            scores_r = scores
        x_in_r  = x_in
        x_out_r = x_out

        T = len(scores_r)
        lo = max(0, T_START - CONTEXT)
        hi = min(T,  T_END   + CONTEXT)

        if lo >= hi:
            print(f"{dataset}: T_START={T_START} >= T_END={T_END} after clipping to [0, {T}). Nothing to plot.")
            continue

        print(f"\n{dataset}: plotting [{lo}:{hi}]  (T_START={T_START}, T_END={T_END}, T={T})")

        # pick top channels by reconstruction error in the core window (T_START:T_END)
        core_lo = min(T_START, T-1)
        core_hi = min(T_END,   T)
        seg_err = np.mean((x_in_r[core_lo:core_hi] - x_out_r[core_lo:core_hi]) ** 2, axis=0)
        top_ch  = np.argsort(seg_err)[-MAX_CH:][::-1]
        print(f"  top channels (highest recon error in [{core_lo}:{core_hi}]): {top_ch.tolist()}")

        t = np.arange(lo, hi)

        n_rows = len(top_ch) + 1
        fig, axes = plt.subplots(n_rows, 1,
                                 figsize=(20, 2.4 * n_rows),
                                 sharex=True, squeeze=False,
                                 gridspec_kw={'height_ratios': [1]*len(top_ch) + [1.4]})
        axes     = axes[:, 0]
        gt_win   = gt_r[lo:hi]
        pred_win = pred_r[lo:hi]

        for row, ch in enumerate(top_ch):
            ax = axes[row]
            shade_spans(ax, gt_win,   lo, C_GT,   alpha=0.20)
            shade_spans(ax, pred_win, lo, C_PRED,  alpha=0.15)
            ax.plot(t, x_in_r[lo:hi,  ch], color=C_INPUT, lw=0.9,
                    label='Input'          if row == 0 else '_')
            ax.plot(t, x_out_r[lo:hi, ch], color=C_RECON, lw=0.9,
                    label='Reconstruction' if row == 0 else '_')
            ax.set_ylabel(f'ch {ch}', fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(True, alpha=0.12)

        ax_s = axes[-1]
        shade_spans(ax_s, gt_win,   lo, C_GT,   alpha=0.20)
        shade_spans(ax_s, pred_win, lo, C_PRED,  alpha=0.15)
        ax_s.plot(t, scores_r[lo:hi], color=C_SCORE, lw=0.9, label='Anomaly score')
        ax_s.axhline(threshold, color=C_THRESH, lw=1.2, ls='--',
                     label=f'Threshold ({threshold:.4g})')
        ax_s.set_yscale('log')
        ax_s.set_ylabel('Score (log)', fontsize=8)
        ax_s.set_xlabel('Time step')
        ax_s.tick_params(labelsize=7)
        ax_s.grid(True, alpha=0.12)

        legend_handles = [
            plt.Line2D([0],[0], color=C_INPUT, lw=1.2, label='Input'),
            plt.Line2D([0],[0], color=C_RECON, lw=1.2, label='Reconstruction'),
            mpatches.Patch(color=C_GT,   alpha=0.35, label='True anomaly'),
            mpatches.Patch(color=C_PRED, alpha=0.35, label='Predicted anomaly'),
        ]
        axes[0].legend(handles=legend_handles, loc='upper right', fontsize=8, ncol=4)

        fig.suptitle(
            f'{dataset} — custom range  [{T_START}:{T_END}]  '
            f'(shown: [{lo}:{hi}] with context={CONTEXT})\n'
            f'Top {len(top_ch)} channels by reconstruction error  '
            f'(blue = true anomaly,  red = predicted anomaly)',
            fontsize=10, y=1.002
        )
        plt.tight_layout()

        out = os.path.join(OUT_DIR, f'{dataset}_custom_{T_START}_{T_END}_channels.png')
        fig.savefig(out, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close(fig)
        print(f"  saved {out}")


SMAP: plotting [63560:67425]  (T_START=63560, T_END=67425, T=427518)
  top channels (highest recon error in [63560:67425]): [0, 5, 6, 3, 19, 1, 21, 18]
  saved ./analysis_plots/SMAP_custom_63560_67425_channels.png


## MSL dataset

In [5]:
!CUDA_VISIBLE_DEVICES=3 python -u run.py \
  --task_name anomaly_detection \
  --is_training 0 \
  --root_path ./dataset/MSL \
  --model_id MSL \
  --model TimesNet \
  --data MSL \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 64 \
  --d_ff 64 \
  --e_layers 3 \
  --enc_in 55 \
  --c_out 55 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 10

Using GPU
Args in experiment:
Basic Config
  Task Name:          anomaly_detection   Is Training:        0                   
  Model ID:           MSL                 Model:              TimesNet            

Data Loader
  Data:               MSL                 Root Path:          ./dataset/MSL       
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Anomaly Detection Task
  Anomaly Ratio:      1.0                 

Model Parameters
  Top k:              3                   Num Kernels:        6                   
  Enc In:             55                  Dec In:             7                   
  C Out:              55                  d model:            64                  
  n heads:            8                   e layers:           3                   
  d layers:           1                   d FF:               64     

## SMAP Dataset

In [ ]:
!CUDA_VISIBLE_DEVICES=2 python -u run.py \
  --task_name anomaly_detection \
  --is_training 0 \
  --root_path ./dataset/SMAP \
  --model_id SMAP \
  --model TimesNet \
  --data SMAP \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 32 \
  --d_ff 32 \
  --e_layers 3 \
  --enc_in 25 \
  --c_out 25 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 10

Using GPU
Args in experiment:
Basic Config
  Task Name:          anomaly_detection   Is Training:        0                   
  Model ID:           SMAP                Model:              TimesNet            

Data Loader
  Data:               SMAP                Root Path:          ./dataset/SMAP      
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Anomaly Detection Task
  Anomaly Ratio:      1.0                 

Model Parameters
  Top k:              3                   Num Kernels:        6                   
  Enc In:             25                  Dec In:             7                   
  C Out:              25                  d model:            32                  
  n heads:            8                   e layers:           3                   
  d layers:           1                   d FF:               32     

## SWaT dataset

In [2]:
!CUDA_VISIBLE_DEVICES=1 python -u run.py \
  --task_name anomaly_detection \
  --is_training 0 \
  --root_path ./dataset/SWaT \
  --model_id SWAT \
  --model TimesNet \
  --data SWAT \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 64 \
  --d_ff 64 \
  --e_layers 3 \
  --enc_in 51 \
  --c_out 51 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 10


Using GPU
Args in experiment:
Basic Config
  Task Name:          anomaly_detection   Is Training:        0                   
  Model ID:           SWAT                Model:              TimesNet            

Data Loader
  Data:               SWAT                Root Path:          ./dataset/SWaT      
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Anomaly Detection Task
  Anomaly Ratio:      1.0                 

Model Parameters
  Top k:              3                   Num Kernels:        6                   
  Enc In:             51                  Dec In:             7                   
  C Out:              51                  d model:            64                  
  n heads:            8                   e layers:           3                   
  d layers:           1                   d FF:               64     

## PSM dataset

In [ ]:
!CUDA_VISIBLE_DEVICES=0 python -u run.py \
  --task_name anomaly_detection \
  --is_training 0 \
  --root_path ./dataset/PSM \
  --model_id PSM \
  --model TimesNet \
  --data PSM \
  --features M \
  --seq_len 100 \
  --pred_len 0 \
  --d_model 32 \
  --d_ff 32 \
  --e_layers 3 \
  --enc_in 25 \
  --c_out 25 \
  --top_k 3 \
  --anomaly_ratio 1 \
  --batch_size 128 \
  --train_epochs 10

## Parameter Count

In [1]:
# ── Parameter count for TimesNet on ALL datasets (best hyperparams) ───
import sys, argparse
import torch

sys.path.insert(0, './')
from models.TimesNet import Model as TimesNet

#datasets = {
#    'SMD':   dict(seq_len=100, enc_in=38, c_out=38, d_model=64,  d_ff=64,  e_layers=2, top_k=5),
#    'MSL':   dict(seq_len=100, enc_in=55, c_out=55, d_model=8,   d_ff=16,  e_layers=1, top_k=3),
#    'SMAP':  dict(seq_len=100, enc_in=25, c_out=25, d_model=128, d_ff=128, e_layers=3, top_k=3),
#    'SWaT':  dict(seq_len=100, enc_in=51, c_out=51, d_model=8,   d_ff=8,   e_layers=3, top_k=3),
#    'PSM':   dict(seq_len=100, enc_in=25, c_out=25, d_model=64,  d_ff=64,  e_layers=2, top_k=3),
#ß}

datasets = {
    # C=38 → 2^6=64 → d_model=64
    'SMD':  dict(seq_len=100, enc_in=38, c_out=38, d_model=64, d_ff=64, e_layers=3, top_k=3),
    # C=55 → 2^6=64 → d_model=64
    'MSL':  dict(seq_len=100, enc_in=55, c_out=55, d_model=64, d_ff=64, e_layers=3, top_k=3),
    # C=25 → 2^5=32 → d_model=32
    'SMAP': dict(seq_len=100, enc_in=25, c_out=25, d_model=32, d_ff=32, e_layers=3, top_k=3),
    # C=51 → 2^6=64 → d_model=64
    'SWaT': dict(seq_len=100, enc_in=51, c_out=51, d_model=64, d_ff=64, e_layers=3, top_k=3),
    # C=25 → 2^5=32 → d_model=32
    'PSM':  dict(seq_len=100, enc_in=25, c_out=25, d_model=32, d_ff=32, e_layers=3, top_k=3),
}

print(f"{'Dataset':<8} {'Total Params':>14} {'Trainable':>14} {'Non-trainable':>15} {'MB (fp32)':>10}")
print("-" * 67)

for name, hp in datasets.items():
    cfg = argparse.Namespace(
        task_name='anomaly_detection',
        label_len=48, pred_len=0,
        num_kernels=6, embed='timeF', freq='h', dropout=0.1,
        **hp,
    )
    model = TimesNet(cfg).cpu().eval()

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    mb_fp32   = total * 4 / 1e6  # params * 4 bytes (float32)

    print(f"{name:<8} {total:>14,} {trainable:>14,} {frozen:>15,} {mb_fp32:>10.2f}")


/home/fzf/.conda/envs/timesnet/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset    Total Params      Trainable   Non-trainable  MB (fp32)
-------------------------------------------------------------------
SMD           7,041,190      7,041,190               0      28.16
MSL           7,045,559      7,045,559               0      28.18
SMAP          1,761,753      1,761,753               0       7.05
SWaT          7,044,531      7,044,531               0      28.18
PSM           1,761,753      1,761,753               0       7.05


In the results, SMD has ~4.7M params because it uses d_model=64, d_ff=64, e_layers=2 (larger hidden dims), while SWaT has only ~112K params because it uses d_model=8, d_ff=8 (much smaller). The parameter count determines how much memory the model needs and is independent of input sequence length — unlike MACs, which scale with seq_len.

## Number of Operations

In [5]:
# ── Count MACs for TimesNet on ALL datasets (best hyperparams) ─────────
import sys, argparse
import torch
import torch.nn as nn

sys.path.insert(0, './')
from models.TimesNet import Model as TimesNet
from fvcore.nn import FlopCountAnalysis
from thop import profile

class _Wrap(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.inner = m
    def forward(self, x):
        return self.inner(x, None, None, None)

#datasets = {
#    # SMD:  F1=0.8459  dm64 df64 el2
#    'SMD':   dict(seq_len=100,  enc_in=38, c_out=38, d_model=64,  d_ff=64,  e_layers=2, top_k=5),
#    # MSL:  F1=0.8180  dm8 df16 el1
#    'MSL':   dict(seq_len=100,  enc_in=55, c_out=55, d_model=8,   d_ff=16,  e_layers=1, top_k=3),
#    # SMAP: F1=0.6944  dm128 df128 el3
#    'SMAP':  dict(seq_len=100,  enc_in=25, c_out=25, d_model=128, d_ff=128, e_layers=3, top_k=3),
#    # SWaT: F1=0.9262  dm8 df8 el3
#    'SWaT':  dict(seq_len=100,  enc_in=51, c_out=51, d_model=8,   d_ff=8,   e_layers=3, top_k=3),
#    # PSM:  F1=0.9738  dm64 df64 el2
#    'PSM':   dict(seq_len=100,  enc_in=25, c_out=25, d_model=64,  d_ff=64,  e_layers=2, top_k=3),
#}

datasets = {
    # C=38 → 2^6=64 → d_model=64
    'SMD':  dict(seq_len=100, enc_in=38, c_out=38, d_model=64, d_ff=64, e_layers=3, top_k=3),
    # C=55 → 2^6=64 → d_model=64
    'MSL':  dict(seq_len=100, enc_in=55, c_out=55, d_model=64, d_ff=64, e_layers=3, top_k=3),
    # C=25 → 2^5=32 → d_model=32
    'SMAP': dict(seq_len=100, enc_in=25, c_out=25, d_model=32, d_ff=32, e_layers=3, top_k=3),
    # C=51 → 2^6=64 → d_model=64
    'SWaT': dict(seq_len=100, enc_in=51, c_out=51, d_model=64, d_ff=64, e_layers=3, top_k=3),
    # C=25 → 2^5=32 → d_model=32
    'PSM':  dict(seq_len=100, enc_in=25, c_out=25, d_model=32, d_ff=32, e_layers=3, top_k=3),
}

print(f"{'Dataset':<8} {'fvcore MACs':>14} {'thop MACs':>14} {'Diff%':>10}")
print("-" * 50)

for name, hp in datasets.items():
    cfg = argparse.Namespace(
        task_name='anomaly_detection',
        label_len=48, pred_len=0,
        num_kernels=6, embed='timeF', freq='h', dropout=0.1,
        **hp,
    )
    model = TimesNet(cfg).cpu().eval()
    wrapped = _Wrap(model)
    torch.manual_seed(42)
    dummy = torch.randn(1, cfg.seq_len, cfg.enc_in)

    with torch.no_grad():
        # fvcore
        fa = FlopCountAnalysis(wrapped, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        fvcore_macs = fa.total()

        # thop
        thop_macs, _ = profile(wrapped, inputs=(dummy,), verbose=False)

    diff_pct = abs(fvcore_macs - thop_macs) / max(fvcore_macs, 1) * 100
    print(f"{name:<8} {fvcore_macs:>14,} {thop_macs:>14,.0f} {diff_pct:>9.4f}%")

Dataset     fvcore MACs      thop MACs      Diff%
--------------------------------------------------
SMD       2,123,747,072  2,123,727,872    0.0009%
MSL       2,110,124,800  2,110,105,600    0.0009%
SMAP        589,610,368    589,600,768    0.0016%
SWaT      2,138,137,344  2,138,118,144    0.0009%
PSM         583,753,088    583,743,488    0.0016%


### Normalizing for streaming data

In [4]:
# ── Count MACs for TimesNet on ALL datasets (best hyperparams) ─────────
import sys, argparse
import torch
import torch.nn as nn
sys.path.insert(0, './')
from models.TimesNet import Model as TimesNet
from fvcore.nn import FlopCountAnalysis
from thop import profile

class _Wrap(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.inner = m
    def forward(self, x):
        return self.inner(x, None, None, None)

#datasets = {
#    'SMD':   dict(seq_len=100, enc_in=38, c_out=38, d_model=64,  d_ff=64,  e_layers=2, top_k=5),
#    'MSL':   dict(seq_len=100, enc_in=55, c_out=55, d_model=8,   d_ff=16,  e_layers=1, top_k=3),
#    'SMAP':  dict(seq_len=100, enc_in=25, c_out=25, d_model=128, d_ff=128, e_layers=3, top_k=3),
#    'SWaT':  dict(seq_len=100, enc_in=51, c_out=51, d_model=8,   d_ff=8,   e_layers=3, top_k=3),
#    'PSM':   dict(seq_len=100, enc_in=25, c_out=25, d_model=64,  d_ff=64,  e_layers=2, top_k=3),
#}

datasets = {
    # C=38 → 2^6=64 → d_model=64
    'SMD':  dict(seq_len=100, enc_in=38, c_out=38, d_model=64, d_ff=64, e_layers=3, top_k=3),
    # C=55 → 2^6=64 → d_model=64
    'MSL':  dict(seq_len=100, enc_in=55, c_out=55, d_model=64, d_ff=64, e_layers=3, top_k=3),
    # C=25 → 2^5=32 → d_model=32
    'SMAP': dict(seq_len=100, enc_in=25, c_out=25, d_model=32, d_ff=32, e_layers=3, top_k=3),
    # C=51 → 2^6=64 → d_model=64
    'SWaT': dict(seq_len=100, enc_in=51, c_out=51, d_model=64, d_ff=64, e_layers=3, top_k=3),
    # C=25 → 2^5=32 → d_model=32
    'PSM':  dict(seq_len=100, enc_in=25, c_out=25, d_model=32, d_ff=32, e_layers=3, top_k=3),
}

print(
    f"{'Dataset':<8} {'seq_len':>8}  "
    f"{'fvcore/sample':>22} {'fvcore/t-point':>22}  "
    f"{'thop/sample':>22} {'thop/t-point':>22}"
)
print("-" * 115)

for name, hp in datasets.items():
    cfg = argparse.Namespace(
        task_name='anomaly_detection',
        label_len=48, pred_len=0,
        num_kernels=6, embed='timeF', freq='h', dropout=0.1,
        **hp,
    )
    model   = TimesNet(cfg).cpu().eval()
    wrapped = _Wrap(model)
    torch.manual_seed(42)

    dummy = torch.randn(1, cfg.seq_len, cfg.enc_in)

    with torch.no_grad():
        fa = FlopCountAnalysis(wrapped, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        fvcore_sample = fa.total()

        thop_sample, _ = profile(wrapped, inputs=(dummy,), verbose=False)

    fvcore_timepoint = fvcore_sample / cfg.seq_len
    thop_timepoint   = thop_sample   / cfg.seq_len

    print(
        f"{name:<8} {cfg.seq_len:>8}  "
        f"{fvcore_sample:>22,} {int(fvcore_timepoint):>22,}  "
        f"{int(thop_sample):>22,} {int(thop_timepoint):>22,}"
    )

print()
print("Columns:")
print("  fvcore/sample  = MACs to process one full sliding window (fvcore)")
print("  fvcore/t-point = effective MACs per new arriving time point (fvcore)")
print("  thop/sample    = MACs to process one full sliding window (thop)")
print("  thop/t-point   = effective MACs per new arriving time point (thop)")
print("  t-point        = MACs/sample ÷ seq_len")

Dataset   seq_len           fvcore/sample         fvcore/t-point             thop/sample           thop/t-point
-------------------------------------------------------------------------------------------------------------------
SMD           100           2,123,747,072             21,237,470           2,123,727,872             21,237,278
MSL           100           2,110,124,800             21,101,248           2,110,105,600             21,101,056
SMAP          100             589,610,368              5,896,103             589,600,768              5,896,007
SWaT          100           2,138,137,344             21,381,373           2,138,118,144             21,381,181
PSM           100             583,753,088              5,837,530             583,743,488              5,837,434

Columns:
  fvcore/sample  = MACs to process one full sliding window (fvcore)
  fvcore/t-point = effective MACs per new arriving time point (fvcore)
  thop/sample    = MACs to process one full sliding window (tho

## Memory Usage

In [1]:
import torch
import torch.nn as nn
import argparse
import sys, os
sys.path.insert(0, "./")
os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # change if needed

from exp.exp_anomaly_detection import Exp_Anomaly_Detection

# ── Force CUDA initialization ────────────────────────────────────────
_ = torch.zeros(1).cuda()
device = torch.device("cuda:0")
print(f"Using device: {torch.cuda.get_device_name(0)}")

DATASETS = {
    # C=38 → 2^6=64 → d_model=64
    "SMD": dict(
        model_id="SMD", data="SMD",
        root_path="./dataset/SMD",
        seq_len=100, enc_in=38, c_out=38,
        d_model=64, d_ff=64, e_layers=3, top_k=3,
        anomaly_ratio=0.5,
    ),
    # C=55 → 2^6=64 → d_model=64
    "MSL": dict(
        model_id="MSL", data="MSL",
        root_path="./dataset/MSL",
        seq_len=100, enc_in=55, c_out=55,
        d_model=64, d_ff=64, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
    # C=25 → 2^5=32 → d_model=32
    "SMAP": dict(
        model_id="SMAP", data="SMAP",
        root_path="./dataset/SMAP",
        seq_len=100, enc_in=25, c_out=25,
        d_model=32, d_ff=32, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
    # C=51 → 2^6=64 → d_model=64
    "SWaT": dict(
        model_id="SWAT", data="SWAT",
        root_path="./dataset/SWaT",
        seq_len=100, enc_in=51, c_out=51,
        d_model=64, d_ff=64, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
    # C=25 → 2^5=32 → d_model=32
    "PSM": dict(
        model_id="PSM", data="PSM",
        root_path="./dataset/PSM",
        seq_len=100, enc_in=25, c_out=25,
        d_model=32, d_ff=32, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
}

# ── Fixed args shared across all datasets ────────────────────────────
BASE_ARGS = dict(
    task_name="anomaly_detection", is_training=0,
    model="TimesNet", data_path="ETTh1.csv",
    features="M", target="OT", freq="h",
    checkpoints="./checkpoints/",
    label_len=48, pred_len=0,
    dec_in=7, n_heads=8, d_layers=1,
    moving_avg=25, factor=1, distil=True,
    dropout=0.1, embed="timeF", activation="gelu",
    num_kernels=6, batch_size=128, train_epochs=10,
    patience=3, learning_rate=0.0001, des="test",
    loss="MSE", lradj="type1", use_amp=False,
    num_workers=10, itr=1, use_gpu=True, gpu=0,
    gpu_type="cuda", use_multi_gpu=False,
    devices="0,1,2,3", p_hidden_dims=[128, 128],
    p_hidden_layers=2, expand=2, d_conv=4,
    output_attention=False,
)

# ── Measure memory for each dataset ──────────────────────────────────
results = {}

for dataset_name, ds_args in DATASETS.items():
    print(f"Measuring {dataset_name}...")
    try:
        # Build args
        args = argparse.Namespace(**{**BASE_ARGS, **ds_args})

        # Clear GPU memory from previous iteration
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        baseline = torch.cuda.memory_allocated(device)

        # Load model
        exp = Exp_Anomaly_Detection(args)
        model = exp.model
        weight_mem = (torch.cuda.memory_allocated(device) - baseline) / 1e6

        # Dummy input matching dataset dimensions
        dummy = torch.randn(
            args.batch_size, args.seq_len, args.enc_in
        ).to(device)
        criterion = nn.MSELoss()

        # Peak training memory (forward + backward)
        torch.cuda.reset_peak_memory_stats(device)
        output = model(dummy, None, None, None)
        loss = criterion(output, dummy)
        loss.backward()
        peak_train = torch.cuda.max_memory_allocated(device) / 1e6

        # Peak inference memory (no gradients)
        torch.cuda.reset_peak_memory_stats(device)
        with torch.no_grad():
            output = model(dummy, None, None, None)
        peak_infer = torch.cuda.max_memory_allocated(device) / 1e6

        total_params = sum(p.numel() for p in model.parameters())

        results[dataset_name] = {
            "params":       total_params,
            "weight_exact": total_params * 4 / 1e6,  # params × 4 bytes → MB
            "weight_mb":    weight_mem,  # includes buffers, varies
            "train_mb":     peak_train,
            "infer_mb":     peak_infer,
            "overhead_mb":  peak_train - weight_mem,
        }

        # Clean up before next dataset
        del model, exp, dummy, output, loss
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"  ERROR: {e}")
        results[dataset_name] = None

# ── Print summary table ───────────────────────────────────────────────
print("\n" + "=" * 95)
print("MEMORY SUMMARY — TimesNet Anomaly Detection (batch_size=128)")
print("=" * 95)
print(f"{'Dataset':<8} {'Params':<12} {'Weights exact':<15} {'Weights GPU':<13} "
      f"{'Train peak':<13} {'Infer peak':<13} {'Overhead':<10}")
print(f"{'':>20} {'(params×4B)':>15} {'(allocated)':>13} "
      f"{'(MB)':>13} {'(MB)':>13} {'(MB)':>10}")
print("-" * 95)
for name, r in results.items():
    if r is None:
        print(f"{name:<8} ERROR")
    else:
        print(f"{name:<8} {r['params']:>10,}   {r['weight_exact']:>11.2f} MB  "
              f"{r['weight_mb']:>9.2f} MB  "
              f"{r['train_mb']:>9.2f} MB  "
              f"{r['infer_mb']:>9.2f} MB  "
              f"{r['overhead_mb']:>7.2f} MB")

/home/fzf/.conda/envs/timesnet/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: NVIDIA GeForce RTX 2080
Measuring SMD...
Use GPU: cuda:0
torch.float32
Measuring MSL...
Use GPU: cuda:0
torch.float32
Measuring SMAP...
Use GPU: cuda:0
torch.float32
Measuring SWaT...
Use GPU: cuda:0
torch.float32
Measuring PSM...
Use GPU: cuda:0
torch.float32

MEMORY SUMMARY — TimesNet Anomaly Detection (batch_size=128)
Dataset  Params       Weights exact   Weights GPU   Train peak    Infer peak    Overhead  
                         (params×4B)   (allocated)          (MB)          (MB)       (MB)
-----------------------------------------------------------------------------------------------
SMD       7,041,190         28.16 MB      29.45 MB     382.10 MB     294.49 MB   352.65 MB
MSL       7,045,559         28.18 MB      29.47 MB     408.93 MB     295.93 MB   379.46 MB
SMAP      1,761,753          7.05 MB       7.70 MB     215.31 MB     101.64 MB   207.61 MB
SWaT      7,044,531         28.18 MB      29.47 MB     407.60 MB     297.28 MB   378.13 MB
PSM       1,761,753   

## Inference Speed

Measures latency, throughput, and s/iter for each dataset using CUDA Events.
- **Latency**: time to process one batch (ms)
- **Throughput**: samples processed per second
- **s/iter**: latency in seconds — comparable to paper Table 11 (Wu et al., 2023)

In [ ]:
import torch
import argparse
import numpy as np
import sys, os
sys.path.insert(0, './')

from exp.exp_anomaly_detection import Exp_Anomaly_Detection

# ── Force CUDA initialization ─────────────────────────────────────────
_ = torch.zeros(1).cuda()
device = torch.device('cuda:0')
print(f"Using device: {torch.cuda.get_device_name(0)}")

DATASETS = {
    # C=38 → 2^6=64 → d_model=64
    "SMD": dict(
        model_id="SMD", data="SMD",
        root_path="./dataset/SMD",
        seq_len=100, enc_in=38, c_out=38,
        d_model=64, d_ff=64, e_layers=3, top_k=3,
        anomaly_ratio=0.5,
    ),
    # C=55 → 2^6=64 → d_model=64
    "MSL": dict(
        model_id="MSL", data="MSL",
        root_path="./dataset/MSL",
        seq_len=100, enc_in=55, c_out=55,
        d_model=64, d_ff=64, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
    # C=25 → 2^5=32 → d_model=32
    "SMAP": dict(
        model_id="SMAP", data="SMAP",
        root_path="./dataset/SMAP",
        seq_len=100, enc_in=25, c_out=25,
        d_model=32, d_ff=32, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
    # C=51 → 2^6=64 → d_model=64
    "SWaT": dict(
        model_id="SWAT", data="SWAT",
        root_path="./dataset/SWaT",
        seq_len=100, enc_in=51, c_out=51,
        d_model=64, d_ff=64, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
    # C=25 → 2^5=32 → d_model=32
    "PSM": dict(
        model_id="PSM", data="PSM",
        root_path="./dataset/PSM",
        seq_len=100, enc_in=25, c_out=25,
        d_model=32, d_ff=32, e_layers=3, top_k=3,
        anomaly_ratio=1.0,
    ),
}

# ── Fixed args shared across all datasets ─────────────────────────────
BASE_ARGS = dict(
    task_name='anomaly_detection', is_training=0,
    model='TimesNet', data_path='ETTh1.csv',
    features='M', target='OT', freq='h',
    checkpoints='./checkpoints/',
    label_len=48, pred_len=0, dec_in=7,
    n_heads=8, d_layers=1, moving_avg=25,
    factor=1, distil=True, dropout=0.1,
    embed='timeF', activation='gelu',
    num_kernels=6, batch_size=128, train_epochs=10,
    patience=3, learning_rate=0.0001, des='test',
    loss='MSE', lradj='type1', use_amp=False,
    num_workers=10, itr=1, use_gpu=True, gpu=0,
    gpu_type='cuda', use_multi_gpu=False,
    devices='0,1,2,3', p_hidden_dims=[128, 128],
    p_hidden_layers=2, expand=2, d_conv=4,
    output_attention=False,
)

N_WARMUP = 10
N_RUNS   = 100

# ── Measure each dataset ──────────────────────────────────────────────
speed_results = {}

for dataset_name, ds_args in DATASETS.items():
    print(f"\nMeasuring {dataset_name}...")
    try:
        args = argparse.Namespace(**{**BASE_ARGS, **ds_args})

        torch.cuda.empty_cache()
        exp   = Exp_Anomaly_Detection(args)
        model = exp.model
        model.eval()

        batch_results = {}

        for batch_size in [1, 128]:
            dummy = torch.randn(batch_size, args.seq_len, args.enc_in).to(device)

            # Warmup
            with torch.no_grad():
                for _ in range(N_WARMUP):
                    _ = model(dummy, None, None, None)

            # Measure
            times_ms = []
            with torch.no_grad():
                for _ in range(N_RUNS):
                    start = torch.cuda.Event(enable_timing=True)
                    end   = torch.cuda.Event(enable_timing=True)
                    torch.cuda.synchronize()
                    start.record()
                    _ = model(dummy, None, None, None)
                    end.record()
                    torch.cuda.synchronize()
                    times_ms.append(start.elapsed_time(end))

            times_ms = np.array(times_ms)
            batch_results[batch_size] = {
                'latency_mean_ms': times_ms.mean(),
                'latency_std_ms':  times_ms.std(),
                'latency_min_ms':  times_ms.min(),
                'latency_max_ms':  times_ms.max(),
                'throughput':      batch_size / (times_ms.mean() / 1000),
                's_per_iter':      times_ms.mean() / 1000,
            }
            print(f"  batch_size={batch_size:<4} "
                  f"latency={times_ms.mean():.3f} ± {times_ms.std():.3f} ms  "
                  f"throughput={batch_size / (times_ms.mean() / 1000):>10,.1f} samp/s  "
                  f"s/iter={times_ms.mean() / 1000:.4f}")

            del dummy

        speed_results[dataset_name] = {
            'seq_len': args.seq_len,
            'enc_in':  args.enc_in,
            'bs1':     batch_results[1],
            'bs128':   batch_results[128],
        }

        del model, exp
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"  ERROR: {e}")
        speed_results[dataset_name] = None

# ── Print summary table ────────────────────────────────────────────────
print("\n" + "=" * 110)
print("INFERENCE SPEED SUMMARY — TimesNet Anomaly Detection (N=100 runs)")
print("=" * 110)
print(f"{'Dataset':<8} {'seq_len':<9} {'enc_in':<8} "
      f"{'bs=1 latency (ms)':<22} {'bs=1 s/iter':<14} "
      f"{'bs=128 latency (ms)':<22} {'bs=128 s/iter':<14} {'bs=128 throughput'}")
print("-" * 110)
for name, r in speed_results.items():
    if r is None:
        print(f"{name:<8} ERROR")
    else:
        lat1   = f"{r['bs1']['latency_mean_ms']:.3f} ± {r['bs1']['latency_std_ms']:.3f}"
        lat128 = f"{r['bs128']['latency_mean_ms']:.3f} ± {r['bs128']['latency_std_ms']:.3f}"
        print(f"{name:<8} {r['seq_len']:<9} {r['enc_in']:<8} "
              f"{lat1:<22} {r['bs1']['s_per_iter']:<14.4f} "
              f"{lat128:<22} {r['bs128']['s_per_iter']:<14.4f} "
              f"{r['bs128']['throughput']:>14,.1f}")
        

Using device: NVIDIA GeForce RTX 2080

Measuring SMD...
Use GPU: cuda:0
  batch_size=1    latency=27.513 ± 0.167 ms  throughput=      36.3 samp/s  s/iter=0.0275
  batch_size=128  latency=140.217 ± 0.779 ms  throughput=     912.9 samp/s  s/iter=0.1402

Measuring MSL...
Use GPU: cuda:0
  batch_size=1    latency=38.483 ± 0.068 ms  throughput=      26.0 samp/s  s/iter=0.0385
  batch_size=128  latency=145.003 ± 2.178 ms  throughput=     882.7 samp/s  s/iter=0.1450

Measuring SMAP...
Use GPU: cuda:0
  batch_size=1    latency=11.956 ± 0.202 ms  throughput=      83.6 samp/s  s/iter=0.0120
  batch_size=128  latency=42.353 ± 0.590 ms  throughput=   3,022.2 samp/s  s/iter=0.0424

Measuring SWaT...
Use GPU: cuda:0
  batch_size=1    latency=45.510 ± 0.106 ms  throughput=      22.0 samp/s  s/iter=0.0455
  batch_size=128  latency=151.274 ± 1.685 ms  throughput=     846.1 samp/s  s/iter=0.1513

Measuring PSM...
Use GPU: cuda:0
  batch_size=1    latency=11.701 ± 0.134 ms  throughput=      85.5 samp/s  